# Proyecto RappiPlus: de datos a decisiones de negocio

**Introducción**


El objetivo de este proyecto es evaluar el desempeño del servicio **RappiPlus** para apoyar **decisiones de negocio basadas en datos**.

Se trabajan con múltiples datasets del negocio:

- **rappiplus_orders_raw.csv** → información de pedidos, precios, descuentos y revenue  
- **rappiplus_catalog.csv** → costos de productos, categorías y proveedores  
- **rappiplus_marketing_spend.csv** → inversión en marketing por canal y país  
- **events / users / user_activity (SQL)** → comportamiento del usuario dentro de la plataforma  
- **experiment_checkout_ui.csv** → resultados de un experimento A/B en el checkout  

El análisis sigue una lógica clara y progresiva:

1. 🔍 Evaluar si podemos confiar en los datos (calidad de datos en Python)

2. 💰 Analizar si el negocio es rentable (revenue, costos y profit)  

3. 🛒 Entender dónde se pierden los usuarios (funnel de conversión)  

4. 🔁 Evaluar si los usuarios regresan (retención por cohortes)  

5. 🧪 Validar si los cambios generan impacto (test estadístico)  

6. 📊 Comunicar los resultados (dashboard en BI)  

A lo largo del proyecto, se transforman datos en insights para responder preguntas clave del negocio y proponer **recomendaciones accionables**.

https://public.tableau.com/app/profile/alexis.lopez7396/viz/sprint12_1/OverviewEjecutivo?publish=yes

https://public.tableau.com/app/profile/alexis.lopez7396/viz/sprint12_2/DashboardDetalle?publish=yes

https://drive.google.com/drive/folders/1mHvNTPu99Bhe9qnN5805FIYZ_a35xca8?usp=sharing

---

## 🔹 Paso 1: Cargar y validar la calidad de los datos

---

### 1.1 Carga de datos y vista rápida

**🎯 Objetivo:** Familiarizarte con la estructura de los datasets del negocio antes de analizarlos.

**Instrucciones:**

- Importa las librerías necesarias
- Carga los archivos:
  - `rappiplus_orders_raw.csv`
  - `rappiplus_catalog.csv`
  - `rappiplus_marketing_spend.csv`
- Guarda los DataFrames en:
  - `orders`, `catalog`, `marketing`
- Explora cada dataset.

---

In [ ]:
# importar librerías
import pandas as pd
import numpy as np
from scipy import stats  # para el test estadístico del Paso 5

In [ ]:
# cargar archivos
# cargar archivos
orders = pd.read_csv('rappiplus_orders_raw.csv')
catalog = pd.read_csv('rappiplus_catalog.csv')
marketing = pd.read_csv('rappiplus_marketing_spend.csv')

In [ ]:
# explorar datasets
print("=== ORDERS ===")
print(orders.shape)
display(orders.head())
orders.info()

=== ORDERS ===
(25100, 12)


,id_pedido,id_usuario,fecha_hora_pedido,pais,dispositivo,fuente_referencia,nombre_producto,categoria_producto,cantidad,precio_unitario,monto_descuento,monto_total
0,order_0,user_6993,2025-05-22,Argentina,desktop,organic,Jacket-Winter-M,Moda,2.0,332.69,0.0,665.37
1,order_1,user_1329,2025-06-15,Mexico,desktop,paid_search,Tablet-Standard-64GB,Electronica,1.0,176.86,5.0,171.86
2,order_2,user_3194,2025-05-02,Argentina,desktop,social,Blender-XL-Red,Hogar,2.0,102.99,10.0,195.99
3,order_3,user_4510,2025-06-09,Colombia,mobile,social,Tablet-Standard-64GB,Electronica,1.0,257.87,15.0,242.87
4,order_4,user_5044,2025-03-30,Argentina,desktop,paid_search,Blender-XL-Red,Hogar,1.0,336.28,0.0,336.28


<class 'pandas.core.frame.DataFrame'>
RangeIndex: 25100 entries, 0 to 25099
Data columns (total 12 columns):
 #   Column              Non-Null Count  Dtype  
---  ------              --------------  -----  
 0   id_pedido           25100 non-null  object 
 1   id_usuario          25100 non-null  object 
 2   fecha_hora_pedido   25100 non-null  object 
 3   pais                24800 non-null  object 
 4   dispositivo         25080 non-null  object 
 5   fuente_referencia   25070 non-null  object 
 6   nombre_producto     25070 non-null  object 
 7   categoria_producto  25020 non-null  object 
 8   cantidad            25050 non-null  float64
 9   precio_unitario     25050 non-null  float64
 10  monto_descuento     25050 non-null  float64
 11  monto_total         25100 non-null  float64
dtypes: float64(4), object(8)
memory usage: 2.3+ MB


In [ ]:
print("=== CATALOG ===")
print(catalog.shape)
display(catalog.head())
catalog.info()

=== CATALOG ===
(7, 4)


,nombre_producto,categoria_producto,costo_unitario,proveedor
0,Laptop-Gaming-16GB,Electrónica,280.68,"Fuller, Pena and Myers"
1,Phone-Pro-128GB,Electrónica,10.12,King Ltd
2,Tablet-Standard-64GB,Electrónica,25.21,Bowers LLC
3,Blender-XL-Red,Hogar,176.64,Long-Reid
4,Vacuum-Pro-Black,Hogar,16.60,"Rivera, Carr and Finley"


<class 'pandas.core.frame.DataFrame'>
RangeIndex: 7 entries, 0 to 6
Data columns (total 4 columns):
 #   Column              Non-Null Count  Dtype  
---  ------              --------------  -----  
 0   nombre_producto     7 non-null      object 
 1   categoria_producto  7 non-null      object 
 2   costo_unitario      7 non-null      float64
 3   proveedor           7 non-null      object 
dtypes: float64(1), object(3)
memory usage: 352.0+ bytes


In [ ]:
print("=== MARKETING ===")
print(marketing.shape)
display(marketing.head())
marketing.info()

=== MARKETING ===
(1620, 5)


,fecha,pais,id_campaña,canal,gasto
0,2025-01-01,Mexico,organic_Mexico,organic,2446.25
1,2025-01-01,Mexico,paid_search_Mexico,paid_search,2704.34
2,2025-01-01,Mexico,social_Mexico,social,2045.01
3,2025-01-01,Colombia,organic_Colombia,organic,2597.21
4,2025-01-01,Colombia,paid_search_Colombia,paid_search,1771.40


<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1620 entries, 0 to 1619
Data columns (total 5 columns):
 #   Column      Non-Null Count  Dtype  
---  ------      --------------  -----  
 0   fecha       1620 non-null   object 
 1   pais        1620 non-null   object 
 2   id_campaña  1620 non-null   object 
 3   canal       1519 non-null   object 
 4   gasto       1620 non-null   float64
dtypes: float64(1), object(4)
memory usage: 63.4+ KB


---

### Revisión y calidad de datos

**🎯 Objetivo:** Detectar y corregir problemas en los datos que puedan afectar el análisis de revenue, costos y rentabilidad.

Se revisan los 3 datasets
- Validar y convertir fechas al formato correcto  
- Revisar variables numéricas (sin negativos o ceros inválidos)  
- Verificar consistencia de montos  
- Eliminar duplicados  
- Revisar variables categóricas

---

In [ ]:
# Limpieza orders
orders = orders.drop_duplicates()
orders['fecha_hora_pedido'] = pd.to_datetime(orders['fecha_hora_pedido'])
orders['pais'] = orders['pais'].str.strip().str.capitalize()
orders = orders[orders['cantidad'] > 0]
orders = orders[orders['monto_total'] > 0]
orders = orders.dropna(subset=['nombre_producto', 'cantidad', 'precio_unitario', 'monto_total'])

print("✅ Orders limpio:", orders.shape)
print("Valores únicos en pais:", orders['pais'].unique())
print("Nulos restantes:\n", orders.isnull().sum())

✅ Orders limpio: (24916, 12)
Valores únicos en pais: ['Argentina' 'Mexico' 'Colombia' nan]
Nulos restantes:
 id_pedido               0
id_usuario              0
fecha_hora_pedido       0
pais                  296
dispositivo            20
fuente_referencia       0
nombre_producto         0
categoria_producto      0
cantidad                0
precio_unitario         0
monto_descuento         0
monto_total             0
dtype: int64


In [ ]:
# Limpieza marketing
marketing['fecha'] = pd.to_datetime(marketing['fecha'])
marketing = marketing.dropna(subset=['canal'])

print("✅ Marketing limpio:", marketing.shape)
print("Canales únicos:", marketing['canal'].unique())

✅ Marketing limpio: (1519, 5)
Canales únicos: ['organic' 'paid_search' 'social']


In [ ]:
# Catalog ya está limpio
print("✅ Catalog sin problemas - nulos:", catalog.isnull().sum().sum(), "| duplicados:", catalog.duplicated().sum())

# Exportar datasets limpios
orders.to_csv('orders_clean.csv', index=False)
catalog.to_csv('catalog_clean.csv', index=False)
marketing.to_csv('marketing_clean.csv', index=False)

print("✅ Archivos exportados correctamente")

✅ Catalog sin problemas - nulos: 0 | duplicados: 0
✅ Archivos exportados correctamente


---
**📦 Exportación**: Una vez finalizada la limpieza, se exportan los datasets para utilizarlos en la última etapa del proyecto.

In [ ]:
# exportar datasets
orders.to_csv('orders_clean.csv', index=False)
catalog.to_csv('catalog_clean.csv', index=False)
marketing.to_csv('marketing_clean.csv', index=False)

---

## 🔹 Paso 2: Analizar si el negocio es rentable

### 2.1 Cálculo de KPIs principales

**🎯 Objetivo:** Calcular los indicadores clave del negocio para evaluar ingresos, costos y rentabilidad.

Se usan los 3 datasets (`orders`, `catalog`, `marketing_spend`):

**📊 Parte 1: Rentabilidad del negocio**
- ¿Cuál es el ingreso total (revenue)?
- ¿Cuál es el costo total?
- ¿Cuánto se ha invertido en marketing?
- ¿El negocio es rentable? (calcular profit)  

---

**📈 Parte 2: Comportamiento de ventas**
- ¿Cuál es el ticket promedio por orden?
- ¿Cuál es la cantidad promedio de productos por orden?
- ¿Cuál es el producto más vendido?
- ¿Cuánto se ha gastado en marketing por canal?

In [ ]:
# PARTE 1: Rentabilidad del negocio

# Unir orders con catalog para obtener costos
orders_catalog = orders.merge(catalog[['nombre_producto', 'costo_unitario']], on='nombre_producto', how='left')

# Calcular costo total por fila
orders_catalog['costo_total'] = orders_catalog['costo_unitario'] * orders_catalog['cantidad']

# KPIs principales
revenue_total = orders_catalog['monto_total'].sum()
costo_total = orders_catalog['costo_total'].sum()
marketing_total = marketing['gasto'].sum()
profit = revenue_total - costo_total - marketing_total

print(f"💰 Revenue total:        ${revenue_total:,.2f}")
print(f"📦 Costo total:          ${costo_total:,.2f}")
print(f"📣 Inversión marketing:  ${marketing_total:,.2f}")
print(f"📈 Profit:               ${profit:,.2f}")
print(f"✅ ¿Es rentable?:        {'Sí' if profit > 0 else 'No'}")


💰 Revenue total:        $51,954,718.94
📦 Costo total:          $43,124,069.01
📣 Inversión marketing:  $2,694,664.43
📈 Profit:               $6,135,985.50
✅ ¿Es rentable?:        Sí


In [ ]:
# PARTE 2: Comportamiento de ventas

# Ticket promedio por orden
ticket_promedio = orders.groupby('id_pedido')['monto_total'].sum().mean()

# Cantidad promedio de productos por orden
cantidad_promedio = orders.groupby('id_pedido')['cantidad'].sum().mean()

# Producto más vendido
producto_mas_vendido = orders.groupby('nombre_producto')['cantidad'].sum().idxmax()

print(f"🛒 Ticket promedio por orden:              ${ticket_promedio:,.2f}")
print(f"📦 Cantidad promedio de productos/orden:   {cantidad_promedio:.2f}")
print(f"🏆 Producto más vendido:                   {producto_mas_vendido}")

🛒 Ticket promedio por orden:              $2,085.20
📦 Cantidad promedio de productos/orden:   7.12
🏆 Producto más vendido:                   Laptop-Gaming-16GB


In [ ]:
# Gasto en marketing por canal
gasto_por_canal = marketing.groupby('canal')['gasto'].sum().reset_index()
gasto_por_canal.columns = ['canal', 'gasto_total']
gasto_por_canal = gasto_por_canal.sort_values('gasto_total', ascending=False)

print("📣 Gasto en marketing por canal:")
display(gasto_por_canal)

📣 Gasto en marketing por canal:


,canal,gasto_total
2,social,918043.21
0,organic,913533.01
1,paid_search,863088.21


---

## 🔹 Paso 3: Entender dónde se pierden los usuarios (funnel de conversión)

**🎯 Objetivo:** Analizar el comportamiento de los usuarios para identificar en qué etapa del proceso se pierden.


⚙️**Conexión a la base de datos**:  
Se ejecuta la línea de configuración para conectar con la base de datos y aplicar consultas SQL en la tabla **events**.

---

**📊 Parte 1: Construcción del funnel**
- ¿Cuántos usuarios llegan a cada etapa del funnel?  
- Se calcula el número de usuarios únicos por `nombre_evento`  
- Se ordenan los eventos según el flujo del usuario  

---

**📉 Parte 2: Análisis de conversión**
- Se calcula la tasa de conversión entre cada paso del funnel  
- Se identifica en qué etapa se pierde la mayor cantidad de usuarios  
- ¿Cuál es la tasa de conversión final?
---

In [ ]:
import pandas as pd
from sqlalchemy import create_engine

# ======================
# Conexión (NO modificar)
# ======================
db_config = {
    'user': 'practicum_student',
    'pwd': 'QnmDH8Sc2TQLvy2G3Vvh7',
    'host': 'yp-trainers-practicum.cluster-czs0gxyx2d8w.us-east-1.rds.amazonaws.com',
    'port': 5432,
    'db': 'data-analyst-production-db-en'
}

connection_string = 'postgresql://{}:{}@{}:{}/{}'.format(
    db_config['user'],
    db_config['pwd'],
    db_config['host'],
    db_config['port'],
    db_config['db']
)

engine = create_engine(connection_string, connect_args={'sslmode':'require'})

In [ ]:
# Explorar tabla events
# =========================
query_events = '''
SELECT *
FROM events;
'''
events = pd.read_sql(query_events, con=engine)
events.head()

,id_usuario,id_sesion,nombre_evento,timestamp_evento,pais,dispositivo,fuente_referencia,categoria_producto
0,user_6772,6a97f2af-32ae-4186-8c92-04025be1a27b,first_visit,2025-05-17,Colombia,desktop,organic,Moda
1,user_5883,369b767c-1c33-4b2f-a652-c7c0ef92cfc9,add_to_cart,2025-02-23,Mexico,mobile,social,Hogar
2,user_5946,60039041-e78b-474c-87b3-c0b7e9c30708,add_payment_info,2025-05-15,Colombia,desktop,social,Electronica
3,user_827,18252a64-f389-4ef7-9e58-dadad4a3491e,purchase,2025-03-31,Mexico,mobile,social,Moda
4,user_2361,221b364e-cdc5-4668-b698-18d5ba849a67,first_visit,2025-01-22,Argentina,desktop,paid_search,Electronica


In [ ]:
# PARTE 1: Totales del funnel
# ======================

query_totals = '''
SELECT
    nombre_evento,
    COUNT(DISTINCT id_usuario) AS usuarios_unicos
FROM events
GROUP BY nombre_evento
ORDER BY usuarios_unicos DESC
'''

totals = pd.read_sql(query_totals, con=engine)
totals

,nombre_evento,usuarios_unicos
0,first_visit,7796
1,add_to_cart,7634
2,select_item,7582
3,begin_checkout,7208
4,add_payment_info,6250
5,purchase,6240


In [ ]:
# PARTE 2: Conversiones
# ======================

query_conversion = '''
SELECT
    nombre_evento,
    COUNT(DISTINCT id_usuario) AS usuarios_unicos,
    ROUND(
        COUNT(DISTINCT id_usuario) * 100.0 / MAX(COUNT(DISTINCT id_usuario)) OVER (),
    2) AS pct_vs_inicio
FROM events
GROUP BY nombre_evento
ORDER BY usuarios_unicos DESC
'''

conversion = pd.read_sql(query_conversion, con=engine)
conversion

,nombre_evento,usuarios_unicos,pct_vs_inicio
0,first_visit,7796,100.00
1,add_to_cart,7634,97.92
2,select_item,7582,97.26
3,begin_checkout,7208,92.46
4,add_payment_info,6250,80.17
5,purchase,6240,80.04


---

## 🔹 Paso 4: Evaluar si los usuarios regresan (retención por cohortes)

**🎯 Objetivo:** Analizar la retención de usuarios para entender si regresan después de registrarse.

**Tablas**

- `users`
- `user_activity`

---
1. Se identifica la cohorte de cada usuario según el **mes de registro**.


2. Se calcula la retención semanal: cuántos usuarios **se mantienen activos** en cada semana desde su registro.
   - `retenido_w1`: usuarios activos en la semana 1  
   - `retenido_w2`: usuarios activos en la semana 2  
   - `retenido_w3`: usuarios activos en la semana 3  


3. Se calcula el porcentaje de retención para cada semana, dividiendo los usuarios retenidos entre los clientes iniciales de la cohorte:  
   - `semana_1`: porcentaje de usuarios retenidos en la semana 1  
   - `semana_2`: porcentaje de usuarios retenidos en la semana 2  
   - `semana_3`: porcentaje de usuarios retenidos en la semana 3  

Se revisa que la columna de fecha esté en formato correcto (`DATE`).  
Se realiza la conversión usando: `CAST(fecha_registro AS DATE)`

In [ ]:
# Explorar tabla users
query_users = '''
SELECT *
FROM users;
'''
users = pd.read_sql(query_users, con=engine)
users.head(3)

,id_usuario,fecha_registro,país,dispositivo,tipo_plan
0,user_0,2025-01-29,Mexico,mobile,free
1,user_1,2025-01-07,Mexico,mobile,free
2,user_2,2025-03-12,Argentina,mobile,free


In [ ]:
# Explorar tabla user_activity
query_user_activity = '''
SELECT *
FROM user_activity;
'''
user_activity = pd.read_sql(query_user_activity, con=engine)
user_activity.head(3)

,id_usuario,fecha_actividad,dias_despues_registro,activo
0,user_0,2025-02-05,7,0
1,user_0,2025-02-12,14,1
2,user_0,2025-02-19,21,1


In [ ]:
# Retención por cohortes
query_cohort_retention_final = '''
SELECT
    TO_CHAR(CAST(u.fecha_registro AS DATE), 'YYYY-MM') AS cohorte,
    COUNT(DISTINCT u.id_usuario) AS clientes_iniciales,
    COUNT(DISTINCT CASE WHEN ua.dias_despues_registro <= 7  AND ua.activo = 1 THEN ua.id_usuario END) AS retenido_w1,
    COUNT(DISTINCT CASE WHEN ua.dias_despues_registro <= 14 AND ua.dias_despues_registro > 7  AND ua.activo = 1 THEN ua.id_usuario END) AS retenido_w2,
    COUNT(DISTINCT CASE WHEN ua.dias_despues_registro <= 21 AND ua.dias_despues_registro > 14 AND ua.activo = 1 THEN ua.id_usuario END) AS retenido_w3,
    ROUND(COUNT(DISTINCT CASE WHEN ua.dias_despues_registro <= 7  AND ua.activo = 1 THEN ua.id_usuario END) * 100.0 / COUNT(DISTINCT u.id_usuario), 2) AS semana_1,
    ROUND(COUNT(DISTINCT CASE WHEN ua.dias_despues_registro <= 14 AND ua.dias_despues_registro > 7  AND ua.activo = 1 THEN ua.id_usuario END) * 100.0 / COUNT(DISTINCT u.id_usuario), 2) AS semana_2,
    ROUND(COUNT(DISTINCT CASE WHEN ua.dias_despues_registro <= 21 AND ua.dias_despues_registro > 14 AND ua.activo = 1 THEN ua.id_usuario END) * 100.0 / COUNT(DISTINCT u.id_usuario), 2) AS semana_3
FROM users u
LEFT JOIN user_activity ua ON u.id_usuario = ua.id_usuario
GROUP BY cohorte
ORDER BY cohorte
'''

# Ejecutar la consulta
cohorte_final = pd.read_sql(query_cohort_retention_final, con=engine)
cohorte_final

,cohorte,clientes_iniciales,retenido_w1,retenido_w2,retenido_w3,semana_1,semana_2,semana_3
0,2025-01,1627,697,668,656,42.84,41.06,40.32
1,2025-02,1444,611,609,635,42.31,42.17,43.98
2,2025-03,1636,677,705,690,41.38,43.09,42.18
3,2025-04,1606,680,697,663,42.34,43.40,41.28
4,2025-05,1687,695,676,706,41.20,40.07,41.85


---

## 🔹 Paso 5: Validar si los cambios generan impacto (test estadístico)

🎯 **Objetivo:** Evaluar si la modificación en la UI del checkout impacta la **tasa de conversión de compra**.

---

1. **Analizar el dataset** `experiment_checkout_ui.csv` para identificar la métrica principal **conversion**.
   - La métrica **conversion** es 1 si el usuario completó la compra, 0 si no.    
2. **Plantear la hipótesis estadística**     
3. **Aplicar el test estadístico adecuado**
4. **Interpretar el resultado**  

---
**Hipótesis estadística**

**H₀** (Hipótesis nula): La tasa de conversión del grupo control es igual a la del grupo tratamiento. El cambio en la UI del checkout no tiene efecto.

**H₁** (Hipótesis alternativa): La tasa de conversión del grupo tratamiento es diferente a la del grupo control. El cambio en la UI del checkout sí tiene efecto.

**Test estadístico:** T-test de dos muestras independientes (scipy.stats.ttest_ind)
Nivel de significancia alpha: 0.05

In [ ]:
# tu código aquí
# Cargar el dataset del experimento
experiment = pd.read_csv('experiment_checkout_ui.csv')

# Separar grupos
control = experiment[experiment['variante'] == 'control']['convirtio']
tratamiento = experiment[experiment['variante'] == 'tratamiento']['convirtio']

# Tasas de conversión
tasa_control = control.mean()
tasa_tratamiento = tratamiento.mean()

print(f"Tasa de conversión Control:     {tasa_control*100:.2f}%")
print(f"Tasa de conversión Tratamiento: {tasa_tratamiento*100:.2f}%")


Tasa de conversión Control:     15.69%
Tasa de conversión Tratamiento: 16.29%


In [ ]:
# Aplicar el test estadístico
stat, p_value = stats.ttest_ind(control, tratamiento)

print(f"Estadístico t: {stat:.4f}")
print(f"P-value:       {p_value:.4f}")
print(f"Alpha:         0.05")

Estadístico t: -0.8132
P-value:       0.4161
Alpha:         0.05


In [ ]:
# Interpretación del resultado
if p_value < 0.05:
    print("Se RECHAZA H0 — hay diferencia significativa entre los grupos.")
    print("El cambio en la UI del checkout SI tiene impacto en la conversion.")
else:
    print("NO se rechaza H0 — no hay diferencia significativa entre los grupos.")
    print("El cambio en la UI del checkout NO tiene impacto en la conversion.")

NO se rechaza H0 — no hay diferencia significativa entre los grupos.
El cambio en la UI del checkout NO tiene impacto en la conversion.


---

## 🔹 Paso 6: Comunicar los resultados (Dashboard en BI)

🎯 **Objetivo**:  
Crear un dashboard que muestre de manera clara y visual los resultados del análisis de ventas, costos, marketing y conversión.

Se usarán los CSVs limpios del Paso 1:

- `orders_clean.csv`  
- `catalog_clean.csv`  
- `marketing_clean.csv`

---

1️⃣ Preparación de los datos
1. Cargar los CSVs en Power BI o Tableau.
2. Revisar relaciones:
   - `orders.nombre_producto` → `catalog.nombre_producto`
   - `orders.fecha_pedido` → tabla de fechas (crear calendario para análisis temporal)
   - `orders.fecha_pedido` → `dim_fecha.date`
3. Crear columnas calculadas necesarias
4. Crear tabla de fechas para poder calcular comparaciones YTD, YoY o períodos anteriores (`Previous Year`, `Previous Month`).

---

2️⃣ Dashboard 1: Overview Ejecutivo
**KPIs principales a mostrar:**
- Revenue total
- Profit total
- Gasto total en marketing
- Ticket promedio
- Cantidad promedio de productos por orden

**Visualizaciones sugeridas:**
- Tarjetas KPI para revenue, profit y gasto marketing
- Gráfico de líneas: evolución mensual de revenue o profit
- Gráfico de líneas YTD
- Gráfico de barras: revenue y profit por producto o categoría

---

 3️⃣ Dashboard 2: Detalle / Drill-through  
**Objetivo:** Permitir explorar los datos desde el KPI general hasta cada orden o producto.

**Visualizaciones sugeridas:**
- Tabla detallada de órdenes con:
  - producto, cantidad, revenue, cost, profit
  - color condicional (profit negativo en rojo, positivo en verde)
- Gráfico de barras por producto con medida `cantidad vendida`
- Drill-through: seleccionar un producto y ver todos los pedidos relacionados
- Filtros por fecha, categoría de producto, etc

---

https://public.tableau.com/app/profile/alexis.lopez7396/viz/sprint12_1/OverviewEjecutivo?publish=yes

https://public.tableau.com/app/profile/alexis.lopez7396/viz/sprint12_2/DashboardDetalle?publish=yes

https://drive.google.com/drive/folders/1mHvNTPu99Bhe9qnN5805FIYZ_a35xca8?usp=sharing